In [1]:
import pandas as pd
import numpy as np
import seaborn.objects as so
import json
from pathlib import Path
from nps import io


## File Structure

in npe2api:
- public/
  - conda/ # specific files for each conda package
  - github/ # branch, issues, PRs, forks, stars, archived, last commit, etc <-- seems quite out of date
  - manifest/ # contains the napari.yaml manifests for each plugin
  - pypi/ # specific files
    - trove classifer
    - author
    - downloads?? maybe
    - release history
    - license
    - requirements
    - requires_python
    - recent release
- classifiers.json # active or withdraw, what pypi versions exist, and the name
- extended_summary.json # summary of:
  - name
  - current version
  - pypi versions
  - author
  - license
  - links
  - seems to exclude withdrawn or deleted plugins

So it looks like I will need to investigate classifiers and extended summary for some basic info, but then crawl specifically each pypi file for more detailed info (like python version, dependences, etc).

In [2]:
NPE2API_DATA = Path("../npe2api/public")
OUTPUT_DIR = Path("reports")
OUTPUT_DIR.mkdir(exist_ok=True)

if NPE2API_DATA.exists():
    pypi_files = list(NPE2API_DATA.glob("pypi/*.json"))
    conda_files = list(NPE2API_DATA.glob("conda/*.json"))
    summary_df = io.load_json_as_dataframe(NPE2API_DATA / "extended_summary.json")
    classifiers_df = io.load_json_as_dataframe(NPE2API_DATA / "classifiers.json")



In [ ]:
status_cols = ['active', 'withdrawn', 'deleted']

classifiers_df[status_cols] = classifiers_df[status_cols].fillna(False).astype(bool)

classifiers_status = classifiers_df[status_cols].idxmax(axis=1)
# classifiers_status[classifiers_status != 'active']  # Show non-active packages
classifiers_status.value_counts()

active       573
withdrawn     33
Name: count, dtype: int64

In [10]:
classifiers_df

,active,withdrawn,deleted,status
acquifer-napari,True,False,False,active
affinder,True,False,False,active
allencell-segmenter-ml,True,False,False,active
alveoleye,True,False,False,active
anchor-droplet-chip,True,False,False,active
...,...,...,...,...
nd2-dask,False,True,False,withdrawn
ome-zarr,False,True,False,withdrawn
pyclesperanto-prototype,False,True,False,withdrawn
tracktour,False,True,False,withdrawn


Set up PyPi data

In [ ]:
pypi_data = {}
for file in pypi_files:
    plugin_name = file.stem
    pypi_data[plugin_name] = io.load_json(file)

pypi_df = pd.DataFrame.from_dict(pypi_data, orient='index')
pypi_df

,info,last_serial,releases,urls,vulnerabilities
acquifer-napari,"{'author': 'Laurent Thomas', 'author_email': '...",22057882,"{'0.0.1': [{'comment_text': '', 'digests': {'b...","[{'comment_text': '', 'digests': {'blake2b_256...",[]
affinder,"{'author': 'Juan Nunez-Iglesias', 'author_emai...",30239145,"{'0.1.0': [{'comment_text': '', 'digests': {'b...","[{'comment_text': None, 'digests': {'blake2b_2...",[]
allencell-segmenter-ml,"{'author': None, 'author_email': None, 'bugtra...",31400556,"{'0.1.16': [{'comment_text': '', 'digests': {'...","[{'comment_text': None, 'digests': {'blake2b_2...",[]
alveoleye,"{'author': 'Joseph Hirsh', 'author_email': 'jo...",30222067,"{'0.1.2': [{'comment_text': '', 'digests': {'b...","[{'comment_text': None, 'digests': {'blake2b_2...",[]
anchor-droplet-chip,"{'author': 'Andrey Aristov', 'author_email': '...",24206867,"{'0.2.0': [{'comment_text': '', 'digests': {'b...","[{'comment_text': '', 'digests': {'blake2b_256...",[]
...,...,...,...,...,...
workshop-demo,"{'author': 'Draga Doncila Pop', 'author_email'...",13143987,"{'0.0.1': [{'comment_text': '', 'digests': {'b...","[{'comment_text': '', 'digests': {'blake2b_256...",[]
world2data,"{'author': 'Marc Boucsein, Robin Koch', 'autho...",12575042,"{'0.0.3': [{'comment_text': '', 'digests': {'b...","[{'comment_text': '', 'digests': {'blake2b_256...",[]
yt-napari,"{'author': 'Chris Havlin', 'author_email': 'ch...",23349457,"{'0.0.1': [{'comment_text': '', 'digests': {'b...","[{'comment_text': '', 'digests': {'blake2b_256...",[]
zarpaint,{'author': 'Abigail S McGovern and Juan Nunez-...,23856378,"{'0.1.0': [{'comment_text': '', 'digests': {'b...","[{'comment_text': '', 'digests': {'blake2b_256...",[]


In [11]:
# add the classifier_df['status'] status to the pypi dataframe
pypi_df['status'] = classifiers_status
pypi_df

,info,last_serial,releases,urls,vulnerabilities,status
acquifer-napari,"{'author': 'Laurent Thomas', 'author_email': '...",22057882,"{'0.0.1': [{'comment_text': '', 'digests': {'b...","[{'comment_text': '', 'digests': {'blake2b_256...",[],active
affinder,"{'author': 'Juan Nunez-Iglesias', 'author_emai...",30239145,"{'0.1.0': [{'comment_text': '', 'digests': {'b...","[{'comment_text': None, 'digests': {'blake2b_2...",[],active
allencell-segmenter-ml,"{'author': None, 'author_email': None, 'bugtra...",31400556,"{'0.1.16': [{'comment_text': '', 'digests': {'...","[{'comment_text': None, 'digests': {'blake2b_2...",[],active
alveoleye,"{'author': 'Joseph Hirsh', 'author_email': 'jo...",30222067,"{'0.1.2': [{'comment_text': '', 'digests': {'b...","[{'comment_text': None, 'digests': {'blake2b_2...",[],active
anchor-droplet-chip,"{'author': 'Andrey Aristov', 'author_email': '...",24206867,"{'0.2.0': [{'comment_text': '', 'digests': {'b...","[{'comment_text': '', 'digests': {'blake2b_256...",[],active
...,...,...,...,...,...,...
workshop-demo,"{'author': 'Draga Doncila Pop', 'author_email'...",13143987,"{'0.0.1': [{'comment_text': '', 'digests': {'b...","[{'comment_text': '', 'digests': {'blake2b_256...",[],active
world2data,"{'author': 'Marc Boucsein, Robin Koch', 'autho...",12575042,"{'0.0.3': [{'comment_text': '', 'digests': {'b...","[{'comment_text': '', 'digests': {'blake2b_256...",[],active
yt-napari,"{'author': 'Chris Havlin', 'author_email': 'ch...",23349457,"{'0.0.1': [{'comment_text': '', 'digests': {'b...","[{'comment_text': '', 'digests': {'blake2b_256...",[],active
zarpaint,{'author': 'Abigail S McGovern and Juan Nunez-...,23856378,"{'0.1.0': [{'comment_text': '', 'digests': {'b...","[{'comment_text': '', 'digests': {'blake2b_256...",[],active


In [12]:
pypi_info = pd.json_normalize(pypi_df['info'])
pypi_info

,author,author_email,bugtrack_url,classifiers,description,description_content_type,docs_url,download_url,dynamic,home_page,...,project_urls.source,project_urls.tracker,project_urls.Citation,project_urls.changelog,project_urls.discussions,project_urls.issues,project_urls.Changelog,project_urls.Report Issues,project_urls.Issue Tracker,project_urls.Bug_Tracker
0,Laurent Thomas,,None,"[Framework :: napari, License :: OSI Approved ...",# acquifer-napari\r\n\r\nThe acquifer-napari p...,text/markdown,None,,None,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Juan Nunez-Iglesias,juan.nunez-iglesias@monash.edu,None,"[Development Status :: 3 - Alpha, Framework ::...",# Description\n\nThis GUI plugin allows you to...,text/markdown,None,None,[License-File],https://github.com/jni/affinder,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,None,None,None,"[Development Status :: 2 - Pre-Alpha, Framewor...",# Allencell-segmenter-ml\n\n[![Test and lint](...,text/markdown,None,None,[License-File],None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Joseph Hirsh,josephhirsh9@gmail.com,None,"[Development Status :: 2 - Pre-Alpha, Framewor...",# AlveolEye: Automated Lung Morphometry Made E...,text/markdown,None,None,[License-File],None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Andrey Aristov,aaristov@pasteur.fr,None,"[Development Status :: 2 - Pre-Alpha, Framewor...",# ⚓ anchor-droplet-chip\n## Measuring single-c...,text/markdown,None,None,None,https://github.com/BaroudLab/anchor-driplet-chip,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
607,Draga Doncila Pop,ddoncilapop@contractor.chanzuckerberg.com,None,"[Development Status :: 2 - Pre-Alpha, Framewor...",# workshop-demo\n\n[![License](https://img.shi...,text/markdown,None,,None,https://github.com/DragaDoncila/workshop-demo,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
608,"Marc Boucsein, Robin Koch",,None,"[Development Status :: 4 - Beta, Framework :: ...",# World2Data\n\n[![License](https://img.shield...,text/markdown,None,,None,https://github.com/MBPhys/World2Data,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
609,Chris Havlin,chris.havlin@gmail.com,None,"[Development Status :: 2 - Pre-Alpha, Framewor...",# yt-napari\n\n[![License](https://img.shields...,text/markdown,None,None,None,https://github.com/data-exp-lab/yt-napari,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
610,Abigail S McGovern and Juan Nunez-Iglesias,juan.nunez-iglesias@monash.edu,None,"[Development Status :: 4 - Beta, Framework :: ...",# zarpaint\n\n[![License](https://img.shields....,text/markdown,None,None,None,https://github.com/jni/zarpaint,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
